# Smart Budget — SageMaker Endpoint Deploy/Test

**Ticket:** DATA-1140  
**Pattern:** SAFE (SKLearnModel)  
**Endpoint name:** `smart-budget-suggestion-endpoint`  
**Bucket:** `s3://blossom-analytics-datalake-dev/smart_budget/endpoint/v1/`

Este notebook empaqueta el pipeline WMA de Smart Budget (loader + aggregator + model) en un
`model.tar.gz`, lo sube a S3 y despliega un endpoint SageMaker on-demand.  
El endpoint recibe `{idaccount, defaultcategory, period_id}` y retorna una sugerencia de presupuesto.

> ⚠️ **Costo**: el endpoint genera costo mientras esté activo. Borrar al terminar (última celda).

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# SETUP — Solo necesario para la sección DEPLOY (Step 1-3)
# Para invocar un endpoint existente (Step 4+) NO ejecutes esta celda.
# ─────────────────────────────────────────────────────────────────────────────
# El SDK clásico de sagemaker (SKLearnModel, get_execution_role) no viene en
# SageMaker Studio. Se instala solo cuando vas a hacer deploy.
# ⚠️  Después de instalar, reinicia el kernel antes de continuar.
# ─────────────────────────────────────────────────────────────────────────────
import subprocess, sys, importlib

def _can_import(module_path):
    try:
        parts = module_path.rsplit('.', 1)
        mod = importlib.import_module(parts[0])
        if len(parts) > 1:
            getattr(mod, parts[1])
        return True
    except (ImportError, AttributeError):
        return False

if not _can_import('sagemaker.sklearn.model.SKLearnModel'):
    print("Instalando sagemaker SDK clásico (solo para deploy)...")
    subprocess.run(
        [sys.executable, "-m", "pip", "install",
         "sagemaker>=2.200,<3", "--quiet",
         "--no-deps",           # evita degradar boto3/botocore
         "--upgrade"],
        check=True
    )
    print("✅ Instalado — REINICIA EL KERNEL (Kernel → Restart Kernel) antes de continuar")
else:
    print("✅ sagemaker SDK listo para deploy")


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# IMPORTS — Sección A: Deploy  (requiere ejecutar la celda Setup primero)
#           Sección B: Invocar (solo boto3, sin sagemaker SDK)
# ─────────────────────────────────────────────────────────────────────────────
import boto3
import json
import os
import shutil
import tarfile
import tempfile
from pathlib import Path

# Sesión boto3 — funciona siempre (Studio o local)
try:
    # Studio: usa las credenciales del entorno de ejecución
    boto_session = boto3.Session()
    region = boto_session.region_name or 'us-east-1'
    # Verificar que las credenciales son válidas
    boto_session.client('sts').get_caller_identity()
    print(f"✅ boto3 listo — región: {region}")
except Exception as e:
    # Local: usa perfil blossom-dev
    boto_session = boto3.Session(profile_name='blossom-dev')
    region = boto_session.region_name or 'us-east-1'
    print(f"✅ Local (blossom-dev) — región: {region}")

# Sección A: imports de sagemaker SDK (solo para deploy)
try:
    import sagemaker
    from sagemaker.sklearn.model import SKLearnModel

    def _get_role():
        try:
            from sagemaker import get_execution_role
            return get_execution_role(sagemaker_session=sagemaker.Session(boto_session=boto_session))
        except Exception:
            sts = boto_session.client('sts')
            arn = sts.get_caller_identity()['Arn']
            if ':assumed-role/' in arn:
                account = arn.split(':')[4]
                role_name = arn.split(':assumed-role/')[1].split('/')[0]
                return f"arn:aws:iam::{account}:role/{role_name}"
            return arn

    sagemaker_session = sagemaker.Session(boto_session=boto_session)
    print("✅ sagemaker SDK disponible (sección Deploy habilitada)")
except ImportError:
    print("ℹ️  sagemaker SDK no disponible — sección Deploy deshabilitada")
    print("   Para invocar endpoints existentes no hace falta. Continúa con Step 4.")


### Step 1: Preparar model.tar.gz

Estructura del tarball:
```
model.tar.gz
├── inference.py           ← entry_point de SageMaker
├── smart_budget/          ← paquete src/smart_budget/
│   ├── __init__.py
│   ├── loader.py
│   ├── model.py
│   ├── aggregator.py
│   └── filters.py
└── data/
    ├── smart_budget_synthetic.csv
    ├── test_internal.csv
    └── test_external.csv
```

In [ ]:
# Paths (relative to repo root — run from repo root or adjust)
REPO_ROOT = Path(os.getcwd()).parent if Path(os.getcwd()).name == 'notebooks' else Path(os.getcwd())

SRC_SMART_BUDGET = REPO_ROOT / 'src' / 'smart_budget'
SRC_INFERENCE = REPO_ROOT / 'src' / 'api' / 'inference.py'
DATA_DIR = REPO_ROOT / 'data' / 'dough'

ARTIFACTS_DIR = REPO_ROOT / 'notebooks' / 'model_artifacts'
ARTIFACTS_DIR.mkdir(exist_ok=True)

# Build staging directory
staging = ARTIFACTS_DIR / 'staging'
if staging.exists():
    shutil.rmtree(staging)
staging.mkdir()

# Copy inference.py as entry_point
shutil.copy(SRC_INFERENCE, staging / 'inference.py')

# Copy smart_budget package
shutil.copytree(SRC_SMART_BUDGET, staging / 'smart_budget')

# Copy data CSVs
data_staging = staging / 'data'
data_staging.mkdir()
for csv_name in ['smart_budget_synthetic.csv']:
    src_file = DATA_DIR / csv_name
    if src_file.exists():
        shutil.copy(src_file, data_staging / csv_name)
for csv_name in ['test_internal.csv', 'test_external.csv']:
    src_file = DATA_DIR / 'test' / csv_name
    if src_file.exists():
        shutil.copy(src_file, data_staging / csv_name)

# Create tarball
tarball_path = ARTIFACTS_DIR / 'model.tar.gz'
with tarfile.open(tarball_path, 'w:gz') as tar:
    for item in staging.rglob('*'):
        if item.is_file():
            arcname = item.relative_to(staging)
            tar.add(item, arcname=arcname)

print(f'model.tar.gz created: {tarball_path}')
print(f'Size: {tarball_path.stat().st_size / 1024:.1f} KB')

### Step 2: Upload model.tar.gz a S3

In [ ]:
S3_BUCKET = 'blossom-analytics-datalake-dev'
S3_KEY = 'smart_budget/endpoint/v1/model.tar.gz'
S3_URI = f's3://{S3_BUCKET}/{S3_KEY}'

s3_client = boto_session.client('s3')
s3_client.upload_file(str(tarball_path), S3_BUCKET, S3_KEY)

print(f'Uploaded to: {S3_URI}')

### Step 3: Deploy con SKLearnModel

In [ ]:
role = get_execution_role(sagemaker_session=sagemaker_session)

sk_model = SKLearnModel(
    model_data=S3_URI,
    role=role,
    entry_point='inference.py',
    framework_version='1.2-1',
    sagemaker_session=sagemaker_session,
)

predictor = sk_model.deploy(
    initial_instance_count=1,
    instance_type='ml.m5.large',
    endpoint_name='smart-budget-suggestion-endpoint',
)

print(f'Endpoint deployed: smart-budget-suggestion-endpoint')

### Step 4: Test del endpoint

In [ ]:
# TC-1: Happy path — cuenta + categoría con datos → suggested_amount > 0
runtime = boto3.client('sagemaker-runtime', region_name=region)

payload = json.dumps({
    'idaccount': 'EXT2',
    'defaultcategory': 'Food & Dining',
    'period_id': '2026-05',
})

response = runtime.invoke_endpoint(
    EndpointName='smart-budget-suggestion-endpoint',
    ContentType='application/json',
    Body=payload,
)
result = json.loads(response['Body'].read().decode('utf-8'))
print("TC-1 Happy path:")
print(json.dumps(result, indent=2))


### Step 5: Verificar campos del response

Verificar que la respuesta contiene los campos esperados del schema acordado.

In [ ]:
# Verificar que la respuesta contiene todos los campos del schema
expected_keys = {
    'idaccount', 'idclient', 'idcompany', 'defaultcategory', 'period_id',
    'suggested_amount', 'confidence', 'basis', 'amount_by_month',
    'display_label', 'model_version',
}
missing = expected_keys - set(result.keys())
if missing:
    print(f'❌ Missing keys: {missing}')
else:
    print('✅ Todos los campos del schema presentes')
    print(f"   suggested_amount : {result['suggested_amount']}")
    print(f"   confidence       : {result['confidence']}")
    print(f"   amount_by_month  : {result.get('amount_by_month')}")
    print(f"   model_version    : {result['model_version']}")

# Validar tipos
assert isinstance(result.get('suggested_amount'), (int, float)) or result['suggested_amount'] is None
assert result.get('model_version') == 'fase0-v1'
print('✅ Tipos válidos')


### Step 4b: Validar las 3 reglas de negocio

| # | Condición | Respuesta esperada |
|---|---|---|
| Regla 1 | `idaccount` no existe | Error (`ModelError`) |
| Regla 2 | `defaultcategory` inválida | Error (`ModelError`) |
| Regla 3 | Cuenta + categoría existen, sin datos | `suggested_amount: null` |


In [ ]:
# Regla 1 — Cuenta no existe → ModelError
import botocore

try:
    response = runtime.invoke_endpoint(
        EndpointName='smart-budget-suggestion-endpoint',
        ContentType='application/json',
        Body=json.dumps({
            'idaccount': 'CUENTA_INEXISTENTE',
            'defaultcategory': 'Groceries',
            'period_id': '2026-05',
        }),
    )
    print("❌ Debería haber fallado")
except runtime.exceptions.ModelError as e:
    print(f"✅ Regla 1 — ModelError recibido: {e.response['Error']['Code']}")
except botocore.exceptions.ClientError as e:
    print(f"✅ Regla 1 — ClientError: {e.response['Error']['Message']}")


In [ ]:
# Regla 2 — Categoría inválida → ModelError
try:
    response = runtime.invoke_endpoint(
        EndpointName='smart-budget-suggestion-endpoint',
        ContentType='application/json',
        Body=json.dumps({
            'idaccount': 'EXT2',
            'defaultcategory': 'CategoriaInexistente',
            'period_id': '2026-05',
        }),
    )
    print("❌ Debería haber fallado")
except runtime.exceptions.ModelError as e:
    print(f"✅ Regla 2 — ModelError recibido: {e.response['Error']['Code']}")
except botocore.exceptions.ClientError as e:
    print(f"✅ Regla 2 — ClientError: {e.response['Error']['Message']}")


In [ ]:
# Regla 3 — Cuenta + categoría existen, sin datos para ese período → null
response = runtime.invoke_endpoint(
    EndpointName='smart-budget-suggestion-endpoint',
    ContentType='application/json',
    Body=json.dumps({
        'idaccount': 'SYN001',
        'defaultcategory': 'Groceries',   # SYN001 no tiene datos en Groceries
        'period_id': '2026-05',
    }),
)
result_null = json.loads(response['Body'].read().decode('utf-8'))
print(f"✅ Regla 3 — suggested_amount: {result_null['suggested_amount']}")
print(f"   display_label: {result_null.get('display_label')}")
assert result_null['suggested_amount'] is None, "Se esperaba null"
print("✅ Regla 3 — null confirmado")


### ! Borrar endpoint (genera costo)

> ⚠️ **Ejecutar esta celda cuando termines.** El endpoint genera costo por hora mientras esté activo.

In [ ]:
# ! Borrar cuando no se use — genera costo
sm_client = boto_session.client('sagemaker')
sm_client.delete_endpoint(EndpointName='smart-budget-suggestion-endpoint')
print('Endpoint deleted.')